# [Notebook]Unihack26_Pipeline_Check

CPU-only software verification. Checks group dataset input hashes, 1,560-D feature layout, masked supervision, gradients, optimizer execution and state-dict round-trip on FABRICATED TENSORS. No real-data accuracy is measured; no release checkpoint is produced. This is a prerequisite to training, not the training experiment itself.

In [ ]:
"""Versioned 1,560-D pair features; statistics may only be fit on training rows."""
from __future__ import annotations

import numpy as np

EMBEDDING_DIM = 384
SCALAR_NAMES = ("cosine", "normalized_match_count", "inlier_ratio", "normalized_reprojection_error",
                "overlap", "roi_coverage", "resolution_ratio", "query_blur", "reference_blur",
                "query_glare", "reference_glare", "view_angle_difference")
FEATURE_DIM = EMBEDDING_DIM * 4 + len(SCALAR_NAMES) * 2
FEATURE_VERSION = "dinov2-s14-pair-v1"


class ScalarNormalizer:
    def __init__(self, mean, scale):
        self.mean = np.asarray(mean, dtype=np.float32)
        self.scale = np.asarray(scale, dtype=np.float32)
        if self.mean.shape != (12,) or self.scale.shape != (12,):
            raise ValueError("EXPECTED_TWELVE_STATISTICS")
        if not np.isfinite(self.mean).all() or not np.isfinite(self.scale).all() or (self.scale <= 0).any():
            raise ValueError("INVALID_STATISTICS")

    @classmethod
    def fit(cls, scalars: np.ndarray, splits: list[str]):
        values = np.asarray(scalars, dtype=np.float32)
        if values.ndim != 2 or values.shape[1] != 12 or len(values) == 0:
            raise ValueError("INVALID_SCALAR_MATRIX")
        if len(splits) != len(values) or any(split != "train" for split in splits):
            raise ValueError("NORMALIZER_TRAIN_ONLY")
        means, scales = [], []
        for column in values.T:
            known = column[np.isfinite(column)]
            means.append(float(known.mean()) if len(known) else 0.0)
            scales.append(max(float(known.std()), 1e-6) if len(known) else 1.0)
        return cls(means, scales)

    def transform(self, values):
        values = np.asarray(values, dtype=np.float32)
        if values.ndim != 2 or values.shape[1] != 12:
            raise ValueError("INVALID_SCALAR_MATRIX")
        missing = ~np.isfinite(values)
        normalized = (np.where(missing, self.mean, values) - self.mean) / self.scale
        return normalized, missing.astype(np.float32)

    def state(self):
        return {"version": FEATURE_VERSION, "scalar_names": list(SCALAR_NAMES),
                "mean": self.mean.tolist(), "scale": self.scale.tolist(), "fit_split": "train"}


def pair_features(query, reference, scalars, normalizer: ScalarNormalizer) -> np.ndarray:
    query, reference = (np.asarray(array, dtype=np.float32) for array in (query, reference))
    if query.ndim != 2 or query.shape[1] != EMBEDDING_DIM or query.shape != reference.shape:
        raise ValueError("INVALID_EMBEDDING_SHAPE")
    if not np.isfinite(query).all() or not np.isfinite(reference).all():
        raise ValueError("NONFINITE_EMBEDDING")
    query_norm, reference_norm = np.linalg.norm(query, axis=1, keepdims=True), np.linalg.norm(reference, axis=1, keepdims=True)
    if (query_norm < 1e-8).any() or (reference_norm < 1e-8).any():
        raise ValueError("ZERO_EMBEDDING")
    query, reference = query / query_norm, reference / reference_norm
    normalized, missing = normalizer.transform(scalars)
    if len(normalized) != len(query):
        raise ValueError("FEATURE_ROW_MISMATCH")
    result = np.concatenate([query, reference, abs(query-reference), query*reference, normalized, missing], axis=1)
    if result.shape[1] != FEATURE_DIM or not np.isfinite(result).all():
        raise ValueError("FEATURE_ASSEMBLY_FAILED")
    return result.astype(np.float32)


In [ ]:
"""Trainable heads for correspondence ranking and conditional evidence sufficiency.

Defining/running these heads is not evidence of quality on real parcel images.
No network downloads or implicit pretrained weights occur in this module.
"""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

INPUT_DIM = 1560


class EvidenceRanker(nn.Module):
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(INPUT_DIM, 256), nn.GELU(), nn.Dropout(dropout),
                                    nn.Linear(256, 64), nn.GELU(), nn.Dropout(dropout))
        self.ranking = nn.Linear(64, 1)
        self.same_region = nn.Linear(64, 1)
        self.sufficiency = nn.Linear(64, 1)

    def forward(self, features):
        if features.ndim != 2 or features.shape[1] != INPUT_DIM:
            raise ValueError("RANKER_INPUT_SHAPE")
        hidden = self.shared(features)
        return {"ranking": self.ranking(hidden).squeeze(-1),
                "same_region": self.same_region(hidden).squeeze(-1),
                "sufficiency": self.sufficiency(hidden).squeeze(-1)}


def masked_bce(logits, targets, mask):
    mask = mask.bool()
    # Empty supervision contributes no gradient while preserving the graph.
    return F.binary_cross_entropy_with_logits(logits[mask], targets[mask]) if mask.any() else logits.sum() * 0


def multitask_loss(outputs, labels, query_ids, ranking_weight=1.0, region_weight=1.0, sufficiency_weight=1.0):
    count = len(outputs["ranking"])
    if len(query_ids) != count or any(len(value) != count for value in labels.values()):
        raise ValueError("LOSS_ROW_MISMATCH")
    region = masked_bce(outputs["same_region"], labels["same_region_target"], labels["same_region_mask"])
    sufficient = labels["sufficiency_mask"].bool() & labels["same_region_mask"].bool() & (labels["same_region_target"] == 1)
    sufficiency = masked_bce(outputs["sufficiency"], labels["sufficiency_target"], sufficient)
    ranking_terms = []
    for query in dict.fromkeys(query_ids):
        indices = torch.tensor([i for i, name in enumerate(query_ids) if name == query], device=outputs["ranking"].device)
        known = labels["same_region_mask"][indices].bool()
        positive = indices[known & (labels["same_region_target"][indices] == 1)]
        negative = indices[known & (labels["same_region_target"][indices] == 0)]
        if len(positive) and len(negative):
            differences = outputs["ranking"][positive, None] - outputs["ranking"][negative][None, :]
            ranking_terms.append(F.softplus(-differences).mean())
    ranking = torch.stack(ranking_terms).mean() if ranking_terms else outputs["ranking"].sum() * 0
    total = ranking_weight * ranking + region_weight * region + sufficiency_weight * sufficiency
    return {"total": total, "ranking": ranking, "same_region": region, "sufficiency": sufficiency,
            "ranked_queries": len(ranking_terms), "sufficiency_rows": int(sufficient.sum())}


def pipeline_check():
    """CPU plumbing tests on fabricated tensors; not an accuracy experiment."""
    torch.manual_seed(26)
    model = EvidenceRanker(dropout=0)
    inputs = torch.randn(8, INPUT_DIM)
    labels = {
        "same_region_target": torch.tensor([1, 0, 1, 0, 1, 0, 0, 0.], dtype=torch.float32),
        "same_region_mask": torch.tensor([1, 1, 1, 1, 1, 1, 0, 0], dtype=torch.bool),
        "sufficiency_target": torch.tensor([1, 0, 0, 0, 1, 0, 0, 0.], dtype=torch.float32),
        "sufficiency_mask": torch.tensor([1, 0, 1, 0, 1, 0, 0, 0], dtype=torch.bool),
    }
    queries = ["q1", "q1", "q2", "q2", "q3", "q3", "q4", "q4"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    initial = float(multitask_loss(model(inputs), labels, queries)["total"].detach())
    for _ in range(12):
        optimizer.zero_grad()
        losses = multitask_loss(model(inputs), labels, queries)
        losses["total"].backward()
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        optimizer.step()
    final = float(multitask_loss(model(inputs), labels, queries)["total"].detach())
    assert final < initial, "OPTIMIZER_DID_NOT_REDUCE_TOY_LOSS"
    unknown = {key: value.clone() for key, value in labels.items()}
    unknown["same_region_mask"][:] = False
    unknown["sufficiency_mask"][:] = False
    assert multitask_loss(model(inputs), unknown, queries)["total"].item() == 0
    with torch.no_grad():
        first = model(inputs)
        restored = EvidenceRanker(dropout=0)
        restored.load_state_dict(model.state_dict())
        for key, value in restored(inputs).items():
            torch.testing.assert_close(value, first[key])
    return {"gate": "PASS", "test_kind": "FABRICATED_TENSOR_PIPELINE_CHECK",
            "real_data_accuracy_measured": False, "trained_release_checkpoint": False,
            "parameter_count": sum(p.numel() for p in model.parameters()),
            "torch_version": torch.__version__, "device": "cpu", "optimizer_steps": 12,
            "initial_toy_loss": initial, "final_toy_loss": final}


In [ ]:
SOURCE_HASHES = {'features': 'bf43387b92a040ef1b2e2528836b73eca79de20eb02ea31962d5180efade217e', 'ranker': '9db0f01016594b3a99217eb86228073dc015722bc251c61fa623ea4a67bfd80f'}
import hashlib, json, platform
from pathlib import Path
torch.set_num_threads(2)
provenance_paths = list(Path('/kaggle/input').rglob('fixture-provenance.json'))
assert len(provenance_paths) == 1, 'TEAM_DATASET_INPUT_MISSING_OR_AMBIGUOUS'
root = provenance_paths[0].parent
provenance = json.loads(provenance_paths[0].read_text())
verified = {}
for filename, expected in provenance['files'].items():
    path = root / filename
    assert path.resolve().is_relative_to(root.resolve())
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected, f'FIXTURE_CHECKSUM_MISMATCH: {filename}'
    verified[filename] = digest
rng = np.random.default_rng(26)
scalars = rng.normal(size=(8, 12)).astype(np.float32)
scalars[:, -1] = np.nan
normalizer = ScalarNormalizer.fit(scalars, ['train'] * 8)
embedding = rng.normal(size=(8, 384)).astype(np.float32)
features = pair_features(embedding, embedding, scalars, normalizer)
assert features.shape == (8, 1560) and np.isfinite(features).all()
assert (features[:, -1] == 1).all()
result = pipeline_check()
result.update(feature_version=FEATURE_VERSION, fixture_hashes=verified,
              numpy_version=np.__version__, python_version=platform.python_version(),
              source_hashes=SOURCE_HASHES, source_dataset='solishu/unihack26-data-protocol',
              source_dataset_version_expected=1)
target = Path('/kaggle/working/pipeline-check.json')
target.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
